# 03 · Train FT-Transformer, TCN and FT-Transformer + TCN
Runtime → **T4 GPU**. Everything is persisted to `models/<kind>/` (last/best weights, `state.json`, predictions, embeddings). If the runtime disconnects: re-run the setup cells, then re-run the model cell — it **resumes from the last finished epoch**, and finished models are skipped.

* `ftt` – FT-Transformer on a single flow
* `tcn` – causal TCN on a window of 10 flows
* `hybrid` – shared FT-Transformer encodes each flow in the window → TCN over the 10 flow embeddings

Windows are 10 consecutive flows of the same day; the target is the label of the last flow. Loss is class-weighted (sqrt-inverse frequency, capped). Best epoch is chosen by **validation (Thursday) attack-vs-benign PR-AUC**, score = 1 − P(BENIGN).

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/NIDS_CICIDS2017_Project')
RAW, EXTRACT, ART, PROC, REP, MODELS = [ROOT/d for d in ['raw','extracted_generated_flows','artifacts','processed','reports','models']]
for d in [RAW, EXTRACT, ART, PROC, REP, MODELS]: d.mkdir(parents=True, exist_ok=True)
SEED = 42

import json, time, gc, math
import numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import average_precision_score
keras.utils.set_random_seed(SEED)
print('GPU:', tf.config.list_physical_devices('GPU'))

SMOKE_TEST = False          # True = 2 tiny epochs to check the pipeline (writes to models/_smoke)
W, BATCH, STEPS, EPOCHS, PATIENCE, VAL_SUB = 10, 512, 1200, 12, 4, 60_000
LR = {'ftt': 5e-4, 'tcn': 1e-3, 'hybrid': 5e-4}
if SMOKE_TEST: STEPS, EPOCHS, VAL_SUB = 30, 2, 5000
OUT = MODELS/'_smoke' if SMOKE_TEST else MODELS

Mounted at /content/drive
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
ARR = PROC/'arrays'
A = {s: {'X': np.load(ARR/f'X_{s}.npy'), 'y': np.load(ARR/f'y_known_{s}.npy'),
         'yb': np.load(ARR/f'y_bin_{s}.npy'), 'day': np.load(ARR/f'day_id_{s}.npy')} for s in ['train','val','test']}
known = json.loads((ART/'label_mapping.json').read_text())['known']; K = len(known); F = A['train']['X'].shape[1]
assert known[0] == 'BENIGN'

def targets(day, W):                      # window end indices whose whole window lies inside one day
    idx = np.arange(W - 1, len(day)); return idx[day[idx - (W - 1)] == day[idx]].astype(np.int64)
T = {s: targets(A[s]['day'], W) for s in A}
print({s: len(t) for s, t in T.items()}, '| features:', F, '| classes:', K)

cnt = np.bincount(A['train']['y'][T['train']], minlength=K).astype('float64')
cw = np.minimum((cnt.sum() / (K * np.maximum(cnt, 1))) ** 0.5, 30.0); cw = (cw / ((cw * cnt).sum() / cnt.sum())).astype('float32')
print(dict(zip(known, np.round(cw, 2))))
VSUB = np.sort(np.random.default_rng(SEED).choice(T['val'], min(VAL_SUB, len(T['val'])), replace=False))

OFFS = tf.constant(np.arange(-(W - 1), 1), tf.int64)
def make_ds(X, y, tidx, seq, batch, shuffle):
    Xc = tf.constant(X); yc = None if y is None else tf.constant(y.astype('int32')); cwc = tf.constant(cw)
    ds = tf.data.Dataset.from_tensor_slices(tidx)
    if shuffle: ds = ds.shuffle(len(tidx), seed=SEED, reshuffle_each_iteration=True).repeat()
    def fetch(i):
        x = tf.gather(Xc, i[:, None] + OFFS[None, :]) if seq else tf.gather(Xc, i)
        if yc is None: return x
        yy = tf.gather(yc, i); return x, yy, tf.gather(cwc, yy)
    return ds.batch(batch).map(fetch, num_parallel_calls=tf.data.AUTOTUNE).prefetch(2)

{'train': 1398371, 'val': 458959, 'test': 703236} | features: 40 | classes: 8
{'BENIGN': np.float32(0.68), 'DoS GoldenEye': np.float32(7.29), 'DoS Hulk': np.float32(1.78), 'DoS Slowhttptest': np.float32(10.23), 'DoS slowloris': np.float32(10.09), 'FTP-Patator': np.float32(9.6), 'Heartbleed': np.float32(53.07), 'SSH-Patator': np.float32(13.04)}


In [3]:
class FeatureTokenizer(layers.Layer):
    # FT-Transformer tokenizer: one learned (w_j, b_j) per feature + a [CLS] token.
    def __init__(self, n_features, d, **kw):
        super().__init__(**kw); self.nf, self.d = n_features, d
    def build(self, shape):
        self.w = self.add_weight(shape=(self.nf, self.d), initializer='glorot_uniform', name='w')
        self.b = self.add_weight(shape=(self.nf, self.d), initializer='zeros', name='b')
        self.cls = self.add_weight(shape=(1, 1, self.d), initializer='glorot_uniform', name='cls')
    def call(self, x):
        tok = x[..., None] * self.w + self.b
        return tf.concat([tf.tile(self.cls, [tf.shape(x)[0], 1, 1]), tok], axis=1)
    def get_config(self): return {**super().get_config(), 'n_features': self.nf, 'd': self.d}

class FTBlock(layers.Layer):
    def __init__(self, d, heads, ff, drop, **kw):
        super().__init__(**kw); self.cfg = (d, heads, ff, drop)
        self.ln1, self.ln2 = layers.LayerNormalization(epsilon=1e-6), layers.LayerNormalization(epsilon=1e-6)
        self.att = layers.MultiHeadAttention(heads, d // heads, dropout=drop)
        self.f1, self.f2, self.dr = layers.Dense(ff * d, activation='gelu'), layers.Dense(d), layers.Dropout(drop)
    def call(self, x, training=None):
        h = self.ln1(x); x = x + self.dr(self.att(h, h, training=training), training=training)
        h = self.ln2(x); return x + self.dr(self.f2(self.dr(self.f1(h), training=training)), training=training)
    def get_config(self):
        d, h, f, dr = self.cfg; return {**super().get_config(), 'd': d, 'heads': h, 'ff': f, 'drop': dr}

def ft_encoder(F, d, heads, nblocks, drop, name):
    inp = keras.Input((F,)); x = FeatureTokenizer(F, d)(inp)
    for _ in range(nblocks): x = FTBlock(d, heads, 2, drop)(x)
    return keras.Model(inp, layers.LayerNormalization()(x[:, 0]), name=name)

def tcn_stack(x, filters=64, k=3, dils=(1, 2, 4), drop=0.1):
    for dl in dils:
        h = x
        for _ in range(2):
            h = layers.Conv1D(filters, k, padding='causal', dilation_rate=dl)(h)
            h = layers.LayerNormalization()(h); h = layers.Activation('relu')(h); h = layers.SpatialDropout1D(drop)(h)
        res = x if x.shape[-1] == filters else layers.Conv1D(filters, 1)(x)
        x = layers.Activation('relu')(layers.Add()([h, res]))
    return x

def build_model(kind):
    if kind == 'ftt':
        inp = keras.Input((F,)); z = ft_encoder(F, 32, 4, 3, 0.1, 'ft_enc')(inp)
    elif kind == 'tcn':
        inp = keras.Input((W, F)); z = tcn_stack(inp)[:, -1]
    else:
        inp = keras.Input((W, F)); z = layers.TimeDistributed(ft_encoder(F, 32, 4, 2, 0.1, 'ft_enc'))(inp)
        z = tcn_stack(z)[:, -1]
    z = layers.Dense(64, activation='relu', name='embedding')(layers.Dropout(0.1)(z))
    return keras.Model(inp, layers.Dense(K, activation='softmax', name='probs')(layers.Dropout(0.1)(z)), name=kind)

for k_ in ['ftt', 'tcn', 'hybrid']: print(k_, f'{build_model(k_).count_params():,} params')

ftt 30,920 params
tcn 77,576 params
hybrid 95,272 params


In [4]:
class ValAPCheckpoint(keras.callbacks.Callback):
    # After every epoch: validation PR-AUC (attack vs benign), save best + last weights and state to Drive, early stopping.
    def __init__(self, ds, yb, mdir, state, patience):
        super().__init__(); self.ds, self.yb, self.mdir, self.s, self.patience = ds, yb, mdir, state, patience
    def _write(self): (self.mdir/'state.json').write_text(json.dumps(self.s))
    def on_epoch_end(self, epoch, logs=None):
        p = self.model.predict(self.ds, verbose=0); ap = float(average_precision_score(self.yb, 1.0 - p[:, 0]))
        if logs is not None: logs['val_ap'] = ap
        s = self.s; s['epoch'] = epoch + 1
        s['history'].append({'epoch': epoch + 1, 'loss': float((logs or {}).get('loss', float('nan'))), 'val_ap': ap})
        if ap > s['best']:
            s.update(best=ap, best_epoch=epoch + 1, wait=0); self.model.save_weights(str(self.mdir/'best.weights.h5'))
        else: s['wait'] += 1
        self.model.save_weights(str(self.mdir/'last.weights.h5'))
        if s['wait'] >= self.patience: self.model.stop_training = True; s['finished'] = True
        self._write(); print(f"  epoch {epoch+1}: val_AP={ap:.4f} (best {s['best']:.4f} @ {s['best_epoch']})")
    def on_train_end(self, logs=None): self.s['finished'] = True; self._write()

def run(kind):
    mdir = OUT/kind; mdir.mkdir(parents=True, exist_ok=True)
    if (mdir/'done.json').exists(): print(kind, 'already finished - skipping'); return
    seq = kind != 'ftt'; keras.backend.clear_session(); gc.collect()
    model = build_model(kind)
    model.compile(optimizer=keras.optimizers.AdamW(LR[kind], weight_decay=1e-4, clipnorm=1.0), loss='sparse_categorical_crossentropy')
    sp = mdir/'state.json'
    state = json.loads(sp.read_text()) if sp.exists() else {'epoch': 0, 'best': -1.0, 'best_epoch': 0, 'wait': 0, 'finished': False, 'history': []}
    if state['epoch'] > 0 and (mdir/'last.weights.h5').exists():
        model.load_weights(str(mdir/'last.weights.h5')); print(f'Resuming {kind} from epoch {state["epoch"]}')
    t0 = time.time()
    if not state['finished'] and state['epoch'] < EPOCHS:
        ds_tr = make_ds(A['train']['X'], A['train']['y'], T['train'], seq, BATCH, True)
        ds_va = make_ds(A['val']['X'], None, VSUB, seq, 2048, False)
        sched = keras.callbacks.LearningRateScheduler(lambda e: LR[kind] * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(e, EPOCHS) / EPOCHS))))
        cb = ValAPCheckpoint(ds_va, A['val']['yb'][VSUB], mdir, state, PATIENCE)
        model.fit(ds_tr, steps_per_epoch=STEPS, epochs=EPOCHS, initial_epoch=state['epoch'], callbacks=[sched, cb], verbose=2)
    model.load_weights(str(mdir/'best.weights.h5'))
    pm = keras.Model(model.input, [model.output, model.get_layer('embedding').output])
    for s in ['val', 'test']:
        pr, em = pm.predict(make_ds(A[s]['X'], None, T[s], seq, 2048, False), verbose=0)
        np.save(mdir/f'{s}_probs.npy', pr.astype('float32')); np.save(mdir/f'{s}_emb.npy', em.astype('float16')); np.save(mdir/f'{s}_tidx.npy', T[s])
    (mdir/'done.json').write_text(json.dumps({'best_epoch': state['best_epoch'], 'best_val_ap': state['best'], 'params': model.count_params(),
                                              'minutes_this_session': round((time.time() - t0) / 60, 1)}))
    print(kind, 'done | best val AP', round(state['best'], 4), '| saved to', mdir)

### Train each model (re-run the same cell after a disconnect to resume)

In [5]:
run('ftt')

Epoch 1/12
  epoch 1: val_AP=0.0119 (best 0.0119 @ 1)
1200/1200 - 62s - 51ms/step - loss: 0.2617 - learning_rate: 5.0000e-04 - val_ap: 0.0119
Epoch 2/12
  epoch 2: val_AP=0.0103 (best 0.0119 @ 1)
1200/1200 - 20s - 17ms/step - loss: 0.0809 - learning_rate: 4.9191e-04 - val_ap: 0.0103
Epoch 3/12
  epoch 3: val_AP=0.0314 (best 0.0314 @ 3)
1200/1200 - 22s - 18ms/step - loss: 0.0570 - learning_rate: 4.6818e-04 - val_ap: 0.0314
Epoch 4/12
  epoch 4: val_AP=0.0253 (best 0.0314 @ 3)
1200/1200 - 21s - 17ms/step - loss: 0.0476 - learning_rate: 4.3044e-04 - val_ap: 0.0253
Epoch 5/12
  epoch 5: val_AP=0.0253 (best 0.0314 @ 3)
1200/1200 - 21s - 17ms/step - loss: 0.0431 - learning_rate: 3.8125e-04 - val_ap: 0.0253
Epoch 6/12
  epoch 6: val_AP=0.0266 (best 0.0314 @ 3)
1200/1200 - 21s - 17ms/step - loss: 0.0372 - learning_rate: 3.2397e-04 - val_ap: 0.0266
Epoch 7/12
  epoch 7: val_AP=0.0143 (best 0.0314 @ 3)
1200/1200 - 21s - 18ms/step - loss: 0.0319 - learning_rate: 2.6250e-04 - val_ap: 0.0143
ftt do

In [6]:
run('tcn')

Epoch 1/12
  epoch 1: val_AP=0.0343 (best 0.0343 @ 1)
1200/1200 - 35s - 29ms/step - loss: 0.0923 - learning_rate: 0.0010 - val_ap: 0.0343
Epoch 2/12
  epoch 2: val_AP=0.0333 (best 0.0343 @ 1)
1200/1200 - 10s - 8ms/step - loss: 0.0291 - learning_rate: 9.8381e-04 - val_ap: 0.0333
Epoch 3/12
  epoch 3: val_AP=0.1032 (best 0.1032 @ 3)
1200/1200 - 10s - 8ms/step - loss: 0.0189 - learning_rate: 9.3636e-04 - val_ap: 0.1032
Epoch 4/12
  epoch 4: val_AP=0.0152 (best 0.1032 @ 3)
1200/1200 - 9s - 7ms/step - loss: 0.0160 - learning_rate: 8.6088e-04 - val_ap: 0.0152
Epoch 5/12
  epoch 5: val_AP=0.0321 (best 0.1032 @ 3)
1200/1200 - 9s - 8ms/step - loss: 0.0146 - learning_rate: 7.6250e-04 - val_ap: 0.0321
Epoch 6/12
  epoch 6: val_AP=0.0366 (best 0.1032 @ 3)
1200/1200 - 9s - 8ms/step - loss: 0.0099 - learning_rate: 6.4794e-04 - val_ap: 0.0366
Epoch 7/12
  epoch 7: val_AP=0.0382 (best 0.1032 @ 3)
1200/1200 - 9s - 7ms/step - loss: 0.0085 - learning_rate: 5.2500e-04 - val_ap: 0.0382
tcn done | best val 

In [7]:
run('hybrid')

Epoch 1/12
  epoch 1: val_AP=0.0286 (best 0.0286 @ 1)
1200/1200 - 291s - 243ms/step - loss: 0.1964 - learning_rate: 5.0000e-04 - val_ap: 0.0286
Epoch 2/12
  epoch 2: val_AP=0.0199 (best 0.0286 @ 1)
1200/1200 - 118s - 98ms/step - loss: 0.0398 - learning_rate: 4.9191e-04 - val_ap: 0.0199
Epoch 3/12
  epoch 3: val_AP=0.0389 (best 0.0389 @ 3)
1200/1200 - 118s - 98ms/step - loss: 0.0257 - learning_rate: 4.6818e-04 - val_ap: 0.0389
Epoch 4/12
  epoch 4: val_AP=0.0194 (best 0.0389 @ 3)
1200/1200 - 117s - 98ms/step - loss: 0.0200 - learning_rate: 4.3044e-04 - val_ap: 0.0194
Epoch 5/12
  epoch 5: val_AP=0.0121 (best 0.0389 @ 3)
1200/1200 - 117s - 98ms/step - loss: 0.0170 - learning_rate: 3.8125e-04 - val_ap: 0.0121
Epoch 6/12


KeyboardInterrupt: 